# Model 2: Qwen2.5-3B + RAG (LangChain + FAISS + Wikipedia DPR KB)
## RAG-Augmented QA with Wikipedia as Knowledge Base (ROUGE, BERTScore & Hallucination Evaluation)

### Design
- **Fine-tuned on**: SQuAD v2 (baseline.ipynb)
- **RAG Knowledge Base**: Wikipedia DPR (100-word passages) — broader coverage
- **Evaluated on**: SQuAD v2 validation split — **same as Model 1**

### Why Wikipedia DPR?
- **Broader coverage**: 18M+ Wikipedia passages vs 80k SQuAD contexts
- **Better chunking**: Pre-optimized 100-word chunks for dense retrieval
- **Domain-diverse**: Not limited to SQuAD training distribution
- **Tests generalization**: Can the model use external knowledge?

```
SQuAD v2 Val Question
        ↓
   FAISS Retriever (Wikipedia DPR KB)
        ↓ top-5 passages (100-word chunks)
   Fine-tuned Qwen2.5-3B
        ↓
   Answer → ROUGE + BERTScore + Faithfulness vs gold answer
```

In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"]   = "1"
print("✅ CUDA debug flags set")

✅ CUDA debug flags set


In [1]:
import torch
import transformers, bitsandbytes, peft, trl

print("📦 Package versions:")
print(f"   torch          : {torch.__version__}")
print(f"   transformers   : {transformers.__version__}")
print(f"   bitsandbytes   : {bitsandbytes.__version__}")
print(f"   peft           : {peft.__version__}")
print(f"   trl            : {trl.__version__}")
print(f"   CUDA           : {torch.version.cuda}")
print()
if torch.cuda.is_available():
    print(f"   Device : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("✅ All checks passed!")

📦 Package versions:
   torch          : 2.7.0+cu128
   transformers   : 4.50.3
   bitsandbytes   : 0.49.2
   peft           : 0.14.0
   trl            : 0.13.0
   CUDA           : 12.8

   Device : NVIDIA GeForce RTX 5060 Laptop GPU
   VRAM   : 8.5 GB
✅ All checks passed!


In [2]:
import os, gc, json, torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document
from langchain.retrievers import BM25Retriever, EnsembleRetriever

from rouge_score import rouge_scorer
from sentence_transformers import CrossEncoder

# ─── CONFIG ─────────────────────────────────────────────────────────────────────────────────
BASE_MODEL          = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_DIR         = "/content/qwen25-squadv2"
MERGED_DIR          = "/content/qwen25-squadv2-merged"
FAISS_PATH          = "./faiss_wikipedia_dpr"          # Wikipedia DPR KB index (changed)
MODEL1_METRICS_PATH = "./model1_outputs/model1_metrics.json"
EMBED_MODEL         = "BAAI/bge-base-en-v1.5"
RERANKER_MODEL      = "cross-encoder/ms-marco-MiniLM-L-6-v2"
SAVE_DIR            = "./model2_outputs_wikipedia"     # Changed for Wikipedia

EVAL_SAMPLES  = 500     # SQuAD v2 validation samples to evaluate on
KB_SAMPLES    = None    # Will use all available Wikipedia passages (loaded from file)
TOP_K         = 5
TOP_K_RERANK  = 10
MAX_SEQ_LEN   = 512
SEED          = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)

print(f"🖥️  Device : {device} — {torch.cuda.get_device_name(0)}")
print(f"   VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB total")
print(f"   Fine-tune base : SQuAD v2 (from baseline.ipynb)")
print(f"   RAG KB         : Wikipedia DPR (100-word passages)")
print(f"   Eval set       : SQuAD v2 validation ({EVAL_SAMPLES} samples)")
print(f"   Embed model    : {EMBED_MODEL}")
print(f"   Retrieval      : BM25 + Dense (Hybrid) ENSEMBLE")
print(f"   Top-K          : {TOP_K}")
print("\n✅ Config ready!")


🖥️  Device : cuda — NVIDIA GeForce RTX 5060 Laptop GPU
   VRAM   : 8.5 GB total
   Fine-tune base : SQuAD v2 (from baseline.ipynb)
   RAG KB         : Wikipedia DPR (100-word passages)
   Eval set       : SQuAD v2 validation (500 samples)
   Embed model    : BAAI/bge-base-en-v1.5
   Retrieval      : BM25 + Dense (Hybrid) ENSEMBLE
   Top-K          : 5

✅ Config ready!


## Load SQuAD v2 (for evaluation) & Wikipedia DPR (for KB)

In [3]:
print("📥 Loading SQuAD v2 (for evaluation)...")
squad = load_dataset("rajpurkar/squad_v2")

for split, d in squad.items():
    print(f"   {split:12s}: {len(d):,} samples")

print("\n📥 Loading Wikipedia DPR KB...")
print("   This downloads ~20GB of Wikipedia passages (one-time)")
print("   Using 'natural-questions' DPR corpus with FAISS index...")

# Download Wikipedia DPR corpus
# Source: https://github.com/facebookresearch/DPR
import urllib.request
import tarfile

dpr_url = "https://dl.fbaipublicfiles.com/dpr/wikipedia_split/psgs_w100.tsv.gz"
dpr_path = "./psgs_w100.tsv.gz"

print(f"\n📥 Downloading Wikipedia DPR passages (~2GB, may take 5-10 min)...")
if not os.path.exists(dpr_path):
    try:
        urllib.request.urlretrieve(dpr_url, dpr_path)
        print("✅ Downloaded")
    except Exception as e:
        print(f"⚠️  Download failed: {e}")
        print("   Falling back to smaller Wikipedia sample...")

# Load Wikipedia passages into documents
print("\n🔨 Loading Wikipedia DPR passages into documents...")
import gzip
import csv

kb_docs = []
doc_id = 0

try:
    with gzip.open(dpr_path, 'rt', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='\t')
        # Skip header
        next(reader)
        
        for row in reader:
            if len(row) < 3:
                continue
            
            passage_id = row[0]
            passage_title = row[1]
            passage_text = row[2]
            
            doc = Document(
                page_content=passage_text,
                metadata={
                    'source': f"wikipedia_dpr_{passage_id}",
                    'title': passage_title,
                    'passage_id': passage_id
                }
            )
            kb_docs.append(doc)
            doc_id += 1
            
            # Progress indicator
            if doc_id % 100000 == 0:
                print(f"   Loaded {doc_id:,} passages...")
                if doc_id >= 500000:  # Limit for faster testing (use all if you have time/storage)
                    print("   (Using 500k passages for faster testing)")
                    break
except Exception as e:
    print(f"⚠️  Error loading Wikipedia: {e}")
    print("   Using fallback: Wikipedia API...")
    
    # Fallback: Use Wikipedia API
    try:
        from langchain.retrievers import WikipediaRetriever
        wiki_retriever = WikipediaRetriever(top_k_results=100, lang="en")
        print("   ✅ Wikipedia API retriever ready (will fetch on-demand)")
    except:
        print("   ❌ Wikipedia API failed. Please install: pip install wikipedia")

if len(kb_docs) == 0:
    print("⚠️  Using Wikipedia API as KB (slower but works)")
else:
    print(f"✅ Loaded {len(kb_docs):,} Wikipedia DPR passages")
    print(f"   Sample passage length: {len(kb_docs[0].page_content)} chars")


📥 Loading SQuAD v2 (for evaluation)...
   train       : 130,319 samples
   validation  : 11,873 samples

📥 Loading Wikipedia DPR KB...
   This downloads ~20GB of Wikipedia passages (one-time)
   Using 'natural-questions' DPR corpus with FAISS index...

📥 Downloading Wikipedia DPR passages (~2GB, may take 5-10 min)...
✅ Downloaded

🔨 Loading Wikipedia DPR passages into documents...
   Loaded 100,000 passages...
   Loaded 200,000 passages...
   Loaded 300,000 passages...
   Loaded 400,000 passages...
   Loaded 500,000 passages...
   (Using 500k passages for faster testing)
✅ Loaded 500,000 Wikipedia DPR passages
   Sample passage length: 5 chars


In [4]:
# Wikipedia DPR is already loaded in previous cell
# No additional KB building needed

print("\n" + "="*70)
print("✅ KNOWLEDGE BASE READY")
print("="*70)
print(f"KB Type: Wikipedia DPR (100-word passages)")
print(f"Coverage: General knowledge + encyclopedic content")
print(f"Advantages: Broader domain coverage, better for OOD questions")
print(f"\nReady for embedding and retrieval setup!")
print("="*70)



✅ KNOWLEDGE BASE READY
KB Type: Wikipedia DPR (100-word passages)
Coverage: General knowledge + encyclopedic content
Advantages: Broader domain coverage, better for OOD questions

Ready for embedding and retrieval setup!


## Build FAISS Vector Store

In [6]:
print("\n" + "="*70)
print("🔨 BUILDING RETRIEVAL SYSTEM (FAISS + BM25 + Ensemble)")
print("="*70)

if len(kb_docs) == 0:
    print("⚠️  KB documents empty, using Wikipedia API instead...")
    # Use Wikipedia API retriever directly
    from langchain_community.retrievers import WikipediaRetriever
    from langchain.retrievers import BM25Retriever
    from langchain.retrievers import EnsembleRetriever
    
    wiki_retriever = WikipediaRetriever(top_k_results=10, lang="en")
    
    print("✅ Wikipedia API retriever created (on-demand retrieval)")
    print("   Note: Will be slower than indexed FAISS, but covers all Wikipedia")
    
    # Note: BM25 won't work with API, so use Wiki directly + Dense ensemble later
    use_api_retriever = True
    dense_retriever = None
    bm25_retriever = None
else:
    print("\n1️⃣  Creating embeddings...")
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-base-en-v1.5",
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True, "batch_size": 128}
    )
    print("   ✅ Embeddings ready")
    
    print("\n2️⃣  Building FAISS index from Wikipedia DPR...")
    print("   (This may take 5-10 min for 500k passages)")
    vectorstore = FAISS.from_documents(kb_docs, embeddings)
    vectorstore.save_local(FAISS_PATH)
    print(f"   ✅ FAISS built and saved to {FAISS_PATH}")
    
    print("\n3️⃣  Building BM25 retriever...")
    bm25_retriever = BM25Retriever.from_documents(kb_docs)
    bm25_retriever.k = 10
    print("   ✅ BM25 built")
    
    print("\n4️⃣  Creating Dense retriever from FAISS...")
    dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
    print("   ✅ Dense retriever created")
    
    print("\n5️⃣  Creating Ensemble retriever...")
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, dense_retriever],
        weights=[0.3, 0.7],  # Will tune this
        k=TOP_K
    )
    print("   ✅ Ensemble created (BM25:0.3 + Dense:0.7)")
    
    retriever = ensemble_retriever
    use_api_retriever = False

print("\n" + "="*70)
print("✅ RETRIEVAL SYSTEM READY")
print("="*70)



🔨 BUILDING RETRIEVAL SYSTEM (FAISS + BM25 + Ensemble)

1️⃣  Creating embeddings...
   ✅ Embeddings ready

2️⃣  Building FAISS index from Wikipedia DPR...
   (This may take 5-10 min for 500k passages)
   ✅ FAISS built and saved to ./faiss_wikipedia_dpr

3️⃣  Building BM25 retriever...
   ✅ BM25 built

4️⃣  Creating Dense retriever from FAISS...
   ✅ Dense retriever created

5️⃣  Creating Ensemble retriever...
   ✅ Ensemble created (BM25:0.3 + Dense:0.7)

✅ RETRIEVAL SYSTEM READY


## Load Fine-Tuned Model (from baseline.ipynb)

In [9]:
# ============================================================
# Initialize Reranker (Missing - Add This)
# ============================================================
print("\n6️⃣  Loading reranker model...")
from sentence_transformers import CrossEncoder

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL, device=device)
print(f"   ✅ Reranker loaded: {RERANKER_MODEL}")

print("\n" + "="*70)
print("✅ ALL SYSTEMS READY")
print("="*70)
print(f"  Embeddings     : {EMBED_MODEL}")
print(f"  Reranker       : {RERANKER_MODEL}")
print(f"  KB Size        : 500,000 Wikipedia passages")
print(f"  VRAM Available : ~4-5 GB")
print("="*70)


6️⃣  Loading reranker model...
   ✅ Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2

✅ ALL SYSTEMS READY
  Embeddings     : BAAI/bge-base-en-v1.5
  Reranker       : cross-encoder/ms-marco-MiniLM-L-6-v2
  KB Size        : 500,000 Wikipedia passages
  VRAM Available : ~4-5 GB


In [7]:
from transformers import BitsAndBytesConfig
import gc

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free before loading: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

print("📥 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

if os.path.exists(MERGED_DIR):
    print(f"📥 Loading pre-merged fine-tuned model from {os.path.abspath(MERGED_DIR)}...")
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR,
        quantization_config=bnb_config,
        device_map="cuda:0",
        trust_remote_code=True,
        attn_implementation="eager",
    )
    model_source = "MERGED_DIR (fine-tuned on SQuAD v2)"
    print("   ✅ Pre-merged model loaded")

elif os.path.exists(ADAPTER_DIR):
    print(f"📥 Loading base model + LoRA adapters from {os.path.abspath(ADAPTER_DIR)}...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="cuda:0",
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        attn_implementation="eager",
    )
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    model = model.merge_and_unload()
    model = model.to(torch.float16)
    model_source = "ADAPTER_DIR (fine-tuned on SQuAD v2, merged on-the-fly)"
    print("   ✅ Adapters merged into base model")

else:
    raise FileNotFoundError(
        f"❌ No fine-tuned model found!\n"
        f"   Expected MERGED_DIR  : {os.path.abspath(MERGED_DIR)}\n"
        f"   Expected ADAPTER_DIR : {os.path.abspath(ADAPTER_DIR)}\n"
        f"   Please run baseline.ipynb first!"
    )

model.config.use_cache = True
model.eval()

print(f"\n✅ Model ready!")
print(f"   Source    : {model_source}")
print(f"   VRAM used : {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"   VRAM free : {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print(f"   dtype     : {next(model.parameters()).dtype}")

VRAM free before loading: 6.38 GB
📥 Loading tokenizer...
📥 Loading pre-merged fine-tuned model from d:\content\qwen25-squadv2-merged...


Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

   ✅ Pre-merged model loaded

✅ Model ready!
   Source    : MERGED_DIR (fine-tuned on SQuAD v2)
   VRAM used : 3.00 GB
   VRAM free : 4.09 GB
   dtype     : torch.float16


## RAG Prompt & Inference Pipeline

In [10]:
from transformers import pipeline as hf_pipeline

gen_pipe = hf_pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
)
print("✅ Generation pipeline ready!")

RAG_SYSTEM = (
    "You are a helpful and precise question-answering assistant. "
    "You are given retrieved passages that may contain relevant information. "
    "Use the context to answer the question accurately and concisely. "
    "If the context does not support an answer, say 'unanswerable'."
)

def retrieve_context(question, k_fetch=TOP_K_RERANK, k_final=TOP_K):
    candidates = vectorstore.similarity_search(question, k=k_fetch)
    pairs      = [[question, doc.page_content] for doc in candidates]
    scores     = reranker.predict(pairs)
    ranked     = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    top_docs   = [doc for _, doc in ranked[:k_final]]
    parts = []
    for i, doc in enumerate(top_docs, 1):
        parts.append(f"[Passage {i}]\n{doc.page_content}")
    return "\n\n".join(parts)

def generate_rag_answer(question, max_new_tokens=150):
    context = retrieve_context(question)
    prompt = (
        f"<|im_start|>system\n{RAG_SYSTEM}<|im_end|>\n"
        f"<|im_start|>user\n"
        f"--- RETRIEVED PASSAGES ---\n{context}\n--- END PASSAGES ---\n\n"
        f"Question: {question}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    out      = gen_pipe(prompt, max_new_tokens=max_new_tokens, do_sample=True,
                        temperature=0.1, top_p=0.9, repetition_penalty=1.1,
                        pad_token_id=tokenizer.eos_token_id)
    full     = out[0]["generated_text"]
    response = full.split("<|im_start|>assistant")[-1]
    response = response.replace("<|im_end|>", "").strip()
    return response, context

# Smoke test on a SQuAD v2 validation sample
print("\n🧪 Testing RAG pipeline...")
test_sample = squad["validation"][0]
ans, ctx    = generate_rag_answer(test_sample["question"])
gold        = test_sample["answers"]["text"][0] if test_sample["answers"]["text"] else "unanswerable"
print(f"Q       : {test_sample['question']}")
print(f"Context : {ctx[:200]}...")
print(f"Answer  : {ans[:200]}")
print(f"Gold    : {gold}")
print("\n✅ RAG pipeline works!")

Device set to use cuda:0


✅ Generation pipeline ready!

🧪 Testing RAG pipeline...
Q       : In what country is Normandy located?
Context : [Passage 1]
Normandy

[Passage 2]
Normandy

[Passage 3]
Normandy

[Passage 4]
Normandy

[Passage 5]
Normandy...
Answer  : France
Gold    : France

✅ RAG pipeline works!


## Full Inference on SQuAD v2 Validation Set

In [11]:
print(f"🔮 RAG inference — {EVAL_SAMPLES} SQuAD v2 validation samples (top-{TOP_K} passages each)...")

rag_results = []
val_subset  = squad["validation"].select(range(EVAL_SAMPLES))

t0 = datetime.now()
for i, sample in enumerate(tqdm(val_subset, desc="RAG Inference")):
    question  = sample["question"]
    answers   = sample["answers"]["text"]
    reference = answers[0] if answers else "unanswerable"
    answerable = len(answers) > 0
    generated, retrieved = generate_rag_answer(question)

    rag_results.append({
        "id":                i,
        "question":          question,
        "generated":         generated,
        "reference":         reference,
        "retrieved_context": retrieved,
        "answerable":        answerable,
    })

rag_df = pd.DataFrame(rag_results)
print(f"\n✅ Done in {datetime.now()-t0}!")
print(f"   Total samples      : {len(rag_df)}")
print(f"   Answerable         : {rag_df['answerable'].sum()}")
print(f"   Unanswerable       : {(~rag_df['answerable']).sum()}")
print(f"\nSample:")
print(f"Q        : {rag_df.iloc[0]['question']}")
print(f"Gold     : {rag_df.iloc[0]['reference']}")
print(f"Generated: {rag_df.iloc[0]['generated'][:200]}")

🔮 RAG inference — 500 SQuAD v2 validation samples (top-5 passages each)...


RAG Inference: 100%|██████████| 500/500 [05:13<00:00,  1.59it/s]


✅ Done in 0:05:13.641532!
   Total samples      : 500
   Answerable         : 237
   Unanswerable       : 263

Sample:
Q        : In what country is Normandy located?
Gold     : France
Generated: France


## ROUGE Evaluation

In [12]:
scorer     = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)
r1, r2, rL = [], [], []

for _, row in tqdm(rag_df.iterrows(), total=len(rag_df), desc="ROUGE"):
    s = scorer.score(row["reference"], row["generated"].strip() or " ")
    r1.append(s["rouge1"].fmeasure)
    r2.append(s["rouge2"].fmeasure)
    rL.append(s["rougeL"].fmeasure)

rag_df["rouge1"] = r1
rag_df["rouge2"] = r2
rag_df["rougeL"] = rL

# Exact match
rag_df["exact_match"] = rag_df.apply(
    lambda row: int(row["generated"].strip().lower() == row["reference"].strip().lower()), axis=1
)

print(f"\n{'='*52}")
print(f"  📊 ROUGE — Model 2 (Qwen2.5-3B + RAG)")
print(f"{'='*52}")
print(f"  ROUGE-1 : {np.mean(r1):.4f}  ±{np.std(r1):.4f}")
print(f"  ROUGE-2 : {np.mean(r2):.4f}  ±{np.std(r2):.4f}")
print(f"  ROUGE-L : {np.mean(rL):.4f}  ±{np.std(rL):.4f}")
print(f"  Exact Match : {rag_df['exact_match'].mean()*100:.2f}%")
print(f"{'='*52}")

ROUGE:   0%|          | 0/500 [00:00<?, ?it/s]

ROUGE: 100%|██████████| 500/500 [00:00<00:00, 9587.81it/s]


  📊 ROUGE — Model 2 (Qwen2.5-3B + RAG)
  ROUGE-1 : 0.4994  ±0.4914
  ROUGE-2 : 0.0078  ±0.0772
  ROUGE-L : 0.4994  ±0.4914
  Exact Match : 48.20%


In [13]:
pip install bert-score

Note: you may need to restart the kernel to use updated packages.


In [14]:
from bert_score import score as bert_score_fn

print("🔮 Computing BERTScore (semantic similarity vs gold answers)...")
print("   Using model: microsoft/deberta-xlarge-mnli")

bs_candidates = rag_df["generated"].apply(lambda x: x.strip() or " ").tolist()
bs_references = rag_df["reference"].tolist()

P, R, F1 = bert_score_fn(
    bs_candidates,
    bs_references,
    lang="en",
    model_type="microsoft/deberta-xlarge-mnli",
    batch_size=16,
    device=device,
    verbose=True,
)

rag_df["bertscore_P"]  = P.numpy()
rag_df["bertscore_R"]  = R.numpy()
rag_df["bertscore_F1"] = F1.numpy()

mean_P  = P.mean().item()
mean_R  = R.mean().item()
mean_F1 = F1.mean().item()

print(f"\n{'='*56}")
print(f"  📊 BERTScore — Model 2 (Qwen2.5-3B + RAG)")
print(f"{'='*56}")
print(f"  Precision : {mean_P:.4f}  ±{P.std().item():.4f}")
print(f"  Recall    : {mean_R:.4f}  ±{R.std().item():.4f}")
print(f"  F1        : {mean_F1:.4f}  ±{F1.std().item():.4f}")
print(f"{'='*56}")

🔮 Computing BERTScore (semantic similarity vs gold answers)...
   Using model: microsoft/deberta-xlarge-mnli
calculating scores...
computing bert embedding.


  0%|          | 0/19 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/32 [00:00<?, ?it/s]

done in 13.04 seconds, 38.33 sentences/sec

  📊 BERTScore — Model 2 (Qwen2.5-3B + RAG)
  Precision : 0.6984  ±0.3049
  Recall    : 0.7137  ±0.2894
  F1        : 0.7041  ±0.2979


## Faithfulness / Hallucination Evaluation

In [15]:
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

print("🧠 Loading NLI model...")
nli = CrossEncoder("cross-encoder/nli-deberta-v3-small", device=device, max_length=512)
print("✅ NLI model ready")

def faith_batch(premises, hypotheses, bs=32):
    scores = []
    for i in range(0, len(premises), bs):
        pairs  = list(zip(premises[i:i+bs], hypotheses[i:i+bs]))
        logits = nli.predict(pairs, apply_softmax=True)
        scores.extend(logits[:, 1].tolist())
    return scores

# Faithfulness vs gold reference answer
print("\n🔮 Computing faithfulness (generated vs gold answer)...")
ref_scores = faith_batch(rag_df["reference"].tolist(), rag_df["generated"].tolist())
rag_df["faith_score"] = ref_scores
rag_df["hallucinated"] = rag_df["faith_score"] < 0.3

# Source grounding vs retrieved context
print("🔮 Computing grounding (generated vs retrieved passages)...")
trunc_ctx  = rag_df["retrieved_context"].apply(lambda x: x[:500]).tolist()
src_scores = faith_batch(trunc_ctx, rag_df["generated"].tolist())
rag_df["src_faith"]  = src_scores
rag_df["src_halluc"] = rag_df["src_faith"] < 0.3

mean_f    = np.mean(ref_scores)
halluc    = rag_df["hallucinated"].mean()
mean_src  = np.mean(src_scores)
halluc_src = rag_df["src_halluc"].mean()

# Per-answerability breakdown
ans_df   = rag_df[rag_df["answerable"] == True]
unans_df = rag_df[rag_df["answerable"] == False]

print(f"\n{'='*56}")
print(f"  🧠 FAITHFULNESS — Model 2 (Qwen2.5-3B + RAG)")
print(f"{'='*56}")
print(f"  Reference faithfulness (vs gold answer):")
print(f"    Mean score        : {mean_f:.4f}")
print(f"    Faithful  (≥0.3)  : {(1-halluc)*100:.1f}%")
print(f"    Hallucinated(<0.3): {halluc*100:.1f}%")
print(f"  Source faithfulness (vs retrieved context):")
print(f"    Mean score        : {mean_src:.4f}")
print(f"    Grounded  (≥0.3)  : {(1-halluc_src)*100:.1f}%")
print(f"    Not grounded(<0.3): {halluc_src*100:.1f}%")
print(f"  Answerable   hallucination rate: {ans_df['hallucinated'].mean()*100:.1f}%")
print(f"  Unanswerable hallucination rate: {unans_df['hallucinated'].mean()*100:.1f}%")
print(f"{'='*56}")

VRAM free: 4.08 GB
🧠 Loading NLI model...
✅ NLI model ready

🔮 Computing faithfulness (generated vs gold answer)...
🔮 Computing grounding (generated vs retrieved passages)...

  🧠 FAITHFULNESS — Model 2 (Qwen2.5-3B + RAG)
  Reference faithfulness (vs gold answer):
    Mean score        : 0.5106
    Faithful  (≥0.3)  : 52.2%
    Hallucinated(<0.3): 47.8%
  Source faithfulness (vs retrieved context):
    Mean score        : 0.0581
    Grounded  (≥0.3)  : 6.0%
    Not grounded(<0.3): 94.0%
  Answerable   hallucination rate: 87.8%
  Unanswerable hallucination rate: 11.8%


## Visualisation — Model 2 (RAG)

In [16]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(18, 16))
fig.suptitle("Model 2: Qwen2.5-3B + RAG (SQuAD v2 KB) — Full Evaluation",
             fontsize=15, fontweight="bold")

# [0,0] ROUGE bars
ax = axes[0, 0]
means = [np.mean(r1), np.mean(r2), np.mean(rL)]
stds  = [np.std(r1),  np.std(r2),  np.std(rL)]
bars  = ax.bar(["ROUGE-1","ROUGE-2","ROUGE-L"], means, yerr=stds,
               color=["#2196F3","#4CAF50","#FF9800"], capsize=5, edgecolor="black", alpha=0.85)
ax.set_ylim(0, 1); ax.set_title("ROUGE Scores", fontweight="bold"); ax.set_ylabel("F1 Score")
ax.axhline(0.4, color="red", linestyle="--", alpha=0.5, label="Good threshold")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
for b, v in zip(bars, means):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.3f}", ha="center", fontweight="bold", fontsize=10)

# [0,1] Faithfulness histogram
ax = axes[0, 1]
ax.hist(ref_scores, bins=20, color="#9C27B0", edgecolor="black", alpha=0.8)
ax.axvline(0.3,    color="red",  linestyle="--", lw=2, label="Hallucination threshold")
ax.axvline(mean_f, color="blue", linestyle="-",  lw=2, label=f"Mean = {mean_f:.3f}")
ax.set_title("Faithfulness Distribution", fontweight="bold")
ax.set_xlabel("Faithfulness Score"); ax.set_ylabel("Count")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

# [0,2] Faithful vs Hallucinated pie
ax = axes[0, 2]
ax.pie([1-halluc, halluc],
       labels=[f"Faithful\n{(1-halluc)*100:.1f}%", f"Hallucinated\n{halluc*100:.1f}%"],
       colors=["#4CAF50","#F44336"], autopct="%1.1f%%", startangle=90,
       wedgeprops={"edgecolor":"white","linewidth":2})
ax.set_title("Faithful vs Hallucinated", fontweight="bold")

# [1,0] Hallucination by answerability
ax = axes[1, 0]
cats = ["Answerable", "Unanswerable"]
halluc_rates = [ans_df["hallucinated"].mean()*100, unans_df["hallucinated"].mean()*100]
bars2 = ax.bar(cats, halluc_rates, color=["#2196F3","#F44336"], edgecolor="black", alpha=0.85, width=0.4)
ax.set_ylim(0, 100); ax.set_title("Hallucination by Answerability", fontweight="bold")
ax.set_ylabel("Hallucination Rate (%)"); ax.grid(axis="y", alpha=0.3)
for b, v in zip(bars2, halluc_rates):
    ax.text(b.get_x()+b.get_width()/2, v+0.8, f"{v:.1f}%", ha="center", fontweight="bold", fontsize=12)

# [1,1] All metrics summary
ax = axes[1, 1]
mnames  = ["Exact Match", "ROUGE-L", "ROUGE-1", "Faithful", "BERTScore F1"]
mvalues = [rag_df["exact_match"].mean()*100, np.mean(rL)*100, np.mean(r1)*100,
           (1-halluc)*100, mean_F1*100]
bars3 = ax.bar(mnames, mvalues,
               color=["#FF5722","#FF9800","#2196F3","#4CAF50","#673AB7"],
               edgecolor="black", alpha=0.85)
ax.set_ylim(0, 100); ax.set_title("All Metrics (%)", fontweight="bold")
ax.set_ylabel("Score (%)"); ax.grid(axis="y", alpha=0.3)
ax.tick_params(axis="x", labelsize=8)
for b, v in zip(bars3, mvalues):
    ax.text(b.get_x()+b.get_width()/2, v+0.8, f"{v:.1f}%", ha="center", fontweight="bold", fontsize=9)

# [1,2] ROUGE-1 by answerability
ax = axes[1, 2]
ans_r1   = rag_df[rag_df["answerable"]==True]["rouge1"].mean()
unans_r1 = rag_df[rag_df["answerable"]==False]["rouge1"].mean()
ax.bar(["Answerable\nROUGE-1", "Unanswerable\nROUGE-1"], [ans_r1, unans_r1],
       color=["#00BCD4","#FF9800"], edgecolor="black", alpha=0.85)
ax.set_ylim(0, 1); ax.set_title("ROUGE-1 by Answerability", fontweight="bold")
ax.set_ylabel("ROUGE-1 F1"); ax.grid(axis="y", alpha=0.3)
for i, v in enumerate([ans_r1, unans_r1]):
    ax.text(i, v+0.01, f"{v:.3f}", ha="center", fontweight="bold", fontsize=12)

# [2,0] BERTScore P/R/F1 bars
ax = axes[2, 0]
bs_means = [mean_P, mean_R, mean_F1]
bs_stds  = [P.std().item(), R.std().item(), F1.std().item()]
bs_bars  = ax.bar(["Precision","Recall","F1"], bs_means, yerr=bs_stds,
                  color=["#3F51B5","#00BCD4","#673AB7"],
                  capsize=5, edgecolor="black", alpha=0.85)
ax.set_ylim(0, 1); ax.set_title("BERTScore (deberta-xlarge-mnli)", fontweight="bold")
ax.set_ylabel("Score")
ax.axhline(0.85, color="red", linestyle="--", alpha=0.5, label="Good threshold (≥0.85)")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
for b, v in zip(bs_bars, bs_means):
    ax.text(b.get_x()+b.get_width()/2, v+0.005, f"{v:.4f}", ha="center", fontweight="bold", fontsize=10)

# [2,1] BERTScore F1 distribution
ax = axes[2, 1]
ax.hist(F1.numpy(), bins=25, color="#673AB7", edgecolor="black", alpha=0.8)
ax.axvline(mean_F1, color="blue", linestyle="-",  lw=2, label=f"Mean = {mean_F1:.4f}")
ax.axvline(0.85,    color="red",  linestyle="--", lw=2, label="Good threshold (0.85)")
ax.set_title("BERTScore F1 Distribution", fontweight="bold")
ax.set_xlabel("BERTScore F1"); ax.set_ylabel("Count")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

# [2,2] Grounding vs retrieved context
ax = axes[2, 2]
ax.hist(src_scores, bins=20, color="#00BCD4", edgecolor="black", alpha=0.8)
ax.axvline(0.3,      color="red",  linestyle="--", lw=2, label="Grounding threshold")
ax.axvline(mean_src, color="blue", linestyle="-",  lw=2, label=f"Mean = {mean_src:.3f}")
ax.set_title("Grounding vs Retrieved Context", fontweight="bold")
ax.set_xlabel("Source Faithfulness Score"); ax.set_ylabel("Count")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
os.makedirs(SAVE_DIR, exist_ok=True)
plt.savefig(f"{SAVE_DIR}/model2_results.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"📊 Saved → {os.path.abspath(SAVE_DIR)}/model2_results.png")

📊 Saved → d:\RAG\model2_outputs_wikipedia/model2_results.png


C:\Users\akhil\AppData\Local\Temp\ipykernel_19568\2966302496.py:108: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Model 1 vs Model 2 — Final Comparison

In [17]:
try:
    with open(MODEL1_METRICS_PATH) as f:
        m1 = json.load(f)
    m1_r1, m1_r2, m1_rL = m1["rouge"]["rouge1"], m1["rouge"]["rouge2"], m1["rouge"]["rougeL"]
    m1_faith  = m1["faithfulness"]["mean_score"]
    m1_halluc = m1["faithfulness"]["hallucination_rate"]
    m1_bs_f1  = m1.get("bertscore", {}).get("f1", None)
    m1_em     = m1.get("exact_match", None)
    print(f"✅ Loaded Model 1 metrics from {os.path.abspath(MODEL1_METRICS_PATH)}")
except FileNotFoundError:
    print(f"⚠️  Not found: {os.path.abspath(MODEL1_METRICS_PATH)}")
    print("   Run baseline.ipynb first — using placeholders for display")
    m1_r1, m1_r2, m1_rL = 0.35, 0.18, 0.30
    m1_faith, m1_halluc  = 0.45, 0.45
    m1_bs_f1, m1_em      = None, None

m2_r1, m2_r2, m2_rL = np.mean(r1), np.mean(r2), np.mean(rL)
m2_faith  = mean_f
m2_halluc = halluc
m2_bs_f1  = mean_F1
m2_em     = rag_df["exact_match"].mean()

print(f"\n{'='*66}")
print(f"  📊 FINAL COMPARISON: Model 1 (No RAG) vs Model 2 (RAG)")
print(f"  Both evaluated on SQuAD v2 validation — clean controlled comparison")
print(f"  Fine-tune: SQuAD v2  |  RAG KB: SQuAD v2 training contexts")
print(f"{'='*66}")
print(f"  {'Metric':<30} {'Model 1':>9} {'Model 2':>9} {'Δ (M2-M1)':>10}")
print(f"  {'─'*60}")

for label, v1, v2 in [
    ("ROUGE-1",            m1_r1,    m2_r1),
    ("ROUGE-2",            m1_r2,    m2_r2),
    ("ROUGE-L",            m1_rL,    m2_rL),
    ("Faithfulness Score", m1_faith, m2_faith),
]:
    print(f"  {label:<30} {v1:>9.4f} {v2:>9.4f} {v2-v1:>+10.4f}")

if m1_bs_f1 is not None:
    print(f"  {'BERTScore F1':<30} {m1_bs_f1:>9.4f} {m2_bs_f1:>9.4f} {m2_bs_f1-m1_bs_f1:>+10.4f}")
else:
    print(f"  {'BERTScore F1':<30} {'N/A':>9} {m2_bs_f1:>9.4f} {'(no M1 baseline)':>10}")

if m1_em is not None:
    print(f"  {'Exact Match':<30} {m1_em*100:>8.1f}% {m2_em*100:>8.1f}% {(m2_em-m1_em)*100:>+9.1f}%")

print(f"  {'─'*60}")
print(f"  {'Hallucination Rate':<30} {m1_halluc*100:>8.1f}% {m2_halluc*100:>8.1f}% {(m2_halluc-m1_halluc)*100:>+9.1f}%")
print(f"{'='*66}")
print(f"  RAG source grounding score  : {mean_src:.4f}")
print(f"  ↑ ROUGE/Faithfulness/BERTScore = better | ↓ Hallucination = better")

✅ Loaded Model 1 metrics from d:\RAG\model1_outputs\model1_metrics.json

  📊 FINAL COMPARISON: Model 1 (No RAG) vs Model 2 (RAG)
  Both evaluated on SQuAD v2 validation — clean controlled comparison
  Fine-tune: SQuAD v2  |  RAG KB: SQuAD v2 training contexts
  Metric                           Model 1   Model 2  Δ (M2-M1)
  ────────────────────────────────────────────────────────────
  ROUGE-1                           0.7629    0.4994    -0.2635
  ROUGE-2                           0.6053    0.0078    -0.5975
  ROUGE-L                           0.7619    0.4994    -0.2625
  Faithfulness Score                0.0506    0.5106    +0.4600
  BERTScore F1                      0.8572    0.7041    -0.1531
  Exact Match                        66.0%     48.2%     -17.8%
  ────────────────────────────────────────────────────────────
  Hallucination Rate                 95.0%     47.8%     -47.2%
  RAG source grounding score  : 0.0581
  ↑ ROUGE/Faithfulness/BERTScore = better | ↓ Hallucination = b

## Visualisation — Full Comparison

In [18]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Model 1 (No RAG) vs Model 2 (RAG) — SQuAD v2 Validation — Controlled Comparison",
             fontsize=14, fontweight="bold")

# [0,0] ROUGE grouped bars
ax = axes[0, 0]
x   = np.arange(3)
w   = 0.35
rouge_labels = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
m1_vals = [m1_r1, m1_r2, m1_rL]
m2_vals = [m2_r1, m2_r2, m2_rL]
b1 = ax.bar(x - w/2, m1_vals, w, label="Model 1 (No RAG)", color="#B0BEC5", edgecolor="black", alpha=0.85)
b2 = ax.bar(x + w/2, m2_vals, w, label="Model 2 (RAG)",    color="#2196F3", edgecolor="black", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(rouge_labels)
ax.set_ylim(0, 1); ax.set_title("ROUGE Scores: M1 vs M2", fontweight="bold")
ax.set_ylabel("F1 Score"); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
for b, v in zip(list(b1)+list(b2), m1_vals+m2_vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.3f}", ha="center", fontsize=8, fontweight="bold")

# [0,1] BERTScore F1 grouped bars
ax = axes[0, 1]
bs_m1 = m1_bs_f1 if m1_bs_f1 is not None else 0
bs_m2 = m2_bs_f1
b1 = ax.bar([0 - 0.2], [bs_m1], 0.35, label="Model 1 (No RAG)", color="#B0BEC5", edgecolor="black", alpha=0.85)
b2 = ax.bar([0 + 0.2], [bs_m2], 0.35, label="Model 2 (RAG)",    color="#673AB7", edgecolor="black", alpha=0.85)
ax.set_xlim(-0.6, 0.6); ax.set_xticks([0]); ax.set_xticklabels(["BERTScore F1"])
ax.set_ylim(0, 1); ax.set_title("BERTScore F1: M1 vs M2", fontweight="bold")
ax.set_ylabel("Score"); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
for b, v in [(b1[0], bs_m1), (b2[0], bs_m2)]:
    ax.text(b.get_x()+b.get_width()/2, v+0.005, f"{v:.4f}", ha="center", fontweight="bold", fontsize=11)

# [0,2] Hallucination rate
ax = axes[0, 2]
cats  = ["Model 1\n(No RAG)", "Model 2\n(RAG)"]
hvals = [m1_halluc*100, m2_halluc*100]
bars2 = ax.bar(cats, hvals, color=["#F44336","#FF9800"], edgecolor="black", alpha=0.85, width=0.4)
ax.set_ylim(0, 100); ax.set_title("Hallucination Rate (lower = better)", fontweight="bold")
ax.set_ylabel("Hallucination Rate (%)"); ax.grid(axis="y", alpha=0.3)
for b, v in zip(bars2, hvals):
    ax.text(b.get_x()+b.get_width()/2, v+0.8, f"{v:.1f}%", ha="center", fontweight="bold", fontsize=12)

# [1,0] Faithfulness comparison
ax = axes[1, 0]
fvals = [m1_faith, m2_faith]
bars3 = ax.bar(cats, fvals, color=["#B0BEC5","#4CAF50"], edgecolor="black", alpha=0.85, width=0.4)
ax.set_ylim(0, 1); ax.set_title("Faithfulness Score: M1 vs M2", fontweight="bold")
ax.set_ylabel("Mean Faithfulness")
ax.axhline(0.3, color="red", linestyle="--", alpha=0.5, label="Threshold")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
for b, v in zip(bars3, fvals):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.4f}", ha="center", fontweight="bold", fontsize=12)

# [1,1] Grounding vs retrieved context (RAG-only metric)
ax = axes[1, 1]
ax.hist(src_scores, bins=20, color="#00BCD4", edgecolor="black", alpha=0.8)
ax.axvline(0.3,      color="red",  linestyle="--", lw=2, label="Grounding threshold")
ax.axvline(mean_src, color="blue", linestyle="-",  lw=2, label=f"Mean = {mean_src:.3f}")
ax.set_title("Model 2: Grounding vs Retrieved Context", fontweight="bold")
ax.set_xlabel("Source Faithfulness Score"); ax.set_ylabel("Count")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

# [1,2] Delta summary (M2 - M1)
ax = axes[1, 2]
delta_labels = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "Faithfulness", "BERTScore F1"]
bs_delta = (m2_bs_f1 - m1_bs_f1) if m1_bs_f1 is not None else 0
deltas   = [m2_r1-m1_r1, m2_r2-m1_r2, m2_rL-m1_rL, m2_faith-m1_faith, bs_delta]
colors   = ["#4CAF50" if d >= 0 else "#F44336" for d in deltas]
bars4    = ax.bar(delta_labels, deltas, color=colors, edgecolor="black", alpha=0.85)
ax.axhline(0, color="black", lw=1)
ax.set_title("Δ RAG Improvement (M2 − M1)", fontweight="bold")
ax.set_ylabel("Delta Score"); ax.grid(axis="y", alpha=0.3)
ax.tick_params(axis="x", labelsize=8)
for b, v in zip(bars4, deltas):
    ypos = v + 0.002 if v >= 0 else v - 0.008
    ax.text(b.get_x()+b.get_width()/2, ypos, f"{v:+.4f}", ha="center", fontweight="bold", fontsize=9)

plt.tight_layout()
os.makedirs(SAVE_DIR, exist_ok=True)
plt.savefig(f"{SAVE_DIR}/model2_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"📊 Saved → {os.path.abspath(SAVE_DIR)}/model2_comparison.png")

📊 Saved → d:\RAG\model2_outputs_wikipedia/model2_comparison.png


C:\Users\akhil\AppData\Local\Temp\ipykernel_19568\2472015481.py:84: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Save Results & Metrics

In [19]:
os.makedirs(SAVE_DIR, exist_ok=True)
rag_df.to_csv(f"{SAVE_DIR}/model2_results.csv", index=False, escapechar="\\")

model2_metrics = {
    "model":            "Qwen2.5-3B-Instruct (Fine-tuned on SQuAD v2 + RAG SQuAD v2 KB)",
    "eval_samples":     EVAL_SAMPLES,
    "kb_samples":       KB_SAMPLES,
    "eval_dataset":     "SQuAD v2 validation",
    "kb_dataset":       "SQuAD v2 training contexts",
    "finetune_dataset": "SQuAD v2",
    "rouge": {
        "rouge1":     round(float(np.mean(r1)), 4),
        "rouge2":     round(float(np.mean(r2)), 4),
        "rougeL":     round(float(np.mean(rL)), 4),
        "rouge1_std": round(float(np.std(r1)),  4),
        "rouge2_std": round(float(np.std(r2)),  4),
        "rougeL_std": round(float(np.std(rL)),  4),
    },
    "exact_match": round(float(rag_df["exact_match"].mean()), 4),
    "bertscore": {
        "precision":     round(mean_P,         4),
        "recall":        round(mean_R,         4),
        "f1":            round(mean_F1,        4),
        "precision_std": round(P.std().item(), 4),
        "recall_std":    round(R.std().item(), 4),
        "f1_std":        round(F1.std().item(),4),
        "model":         "microsoft/deberta-xlarge-mnli",
    },
    "faithfulness": {
        "mean_score":          round(float(mean_f),    4),
        "hallucination_rate":  round(float(halluc),    4),
        "faithful_pct":        round((1-halluc)*100,   2),
        "halluc_pct":          round(halluc*100,       2),
        "source_mean":         round(float(mean_src),  4),
        "source_grounded_pct": round((1-halluc_src)*100, 2),
        "samples_evaluated":   int(len(rag_df)),
    },
}

with open(f"{SAVE_DIR}/model2_metrics.json", "w") as f:
    json.dump(model2_metrics, f, indent=2)

bs_delta = round(m2_bs_f1 - m1_bs_f1, 4) if m1_bs_f1 is not None else None
comparison = {
    "experiment_note": "Clean controlled comparison — both models evaluated on SQuAD v2 validation",
    "model1_no_rag": {
        "dataset":      "SQuAD v2 validation",
        "rouge1": m1_r1, "rouge2": m1_r2, "rougeL": m1_rL,
        "faithfulness": m1_faith, "hallucination": m1_halluc,
        "bertscore_f1": m1_bs_f1,
    },
    "model2_rag": {
        "finetune_dataset": "SQuAD v2",
        "kb_dataset":       "SQuAD v2 training contexts",
        "eval_dataset":     "SQuAD v2 validation",
        "rouge1":        round(m2_r1,    4),
        "rouge2":        round(m2_r2,    4),
        "rougeL":        round(m2_rL,    4),
        "faithfulness":  round(m2_faith, 4),
        "hallucination": round(m2_halluc,4),
        "bertscore_f1":  round(m2_bs_f1, 4),
    },
    "delta_model2_minus_model1": {
        "rouge1":        round(m2_r1    - m1_r1,    4),
        "rouge2":        round(m2_r2    - m1_r2,    4),
        "rougeL":        round(m2_rL    - m1_rL,    4),
        "faithfulness":  round(m2_faith - m1_faith, 4),
        "hallucination": round(m2_halluc- m1_halluc,4),
        "bertscore_f1":  bs_delta,
    },
    "note": "Positive delta = Model 2 better for ROUGE/faithfulness/BERTScore. Negative delta = Model 2 better for hallucination."
}

with open(f"{SAVE_DIR}/final_comparison.json", "w") as f:
    json.dump(comparison, f, indent=2)

print(f"💾 Saved to: {os.path.abspath(SAVE_DIR)}")
print(f"   model2_results.csv")
print(f"   model2_metrics.json")
print(f"   final_comparison.json")
print()
print("=" * 52)
print("  ✅ MODEL 2 COMPLETE — SUMMARY")
print("=" * 52)
print(f"  ROUGE-1       : {model2_metrics['rouge']['rouge1']:.4f}")
print(f"  ROUGE-2       : {model2_metrics['rouge']['rouge2']:.4f}")
print(f"  ROUGE-L       : {model2_metrics['rouge']['rougeL']:.4f}")
print(f"  BERTScore F1  : {model2_metrics['bertscore']['f1']:.4f}")
print(f"  Faithful      : {model2_metrics['faithfulness']['faithful_pct']:.1f}%")
print(f"  Hallucinate   : {model2_metrics['faithfulness']['halluc_pct']:.1f}%")
print(f"  Grounded      : {model2_metrics['faithfulness']['source_grounded_pct']:.1f}%")
print("=" * 52)

💾 Saved to: d:\RAG\model2_outputs_wikipedia
   model2_results.csv
   model2_metrics.json
   final_comparison.json

  ✅ MODEL 2 COMPLETE — SUMMARY
  ROUGE-1       : 0.4994
  ROUGE-2       : 0.0078
  ROUGE-L       : 0.4994
  BERTScore F1  : 0.7041
  Faithful      : 52.2%
  Hallucinate   : 47.8%
  Grounded      : 6.0%


In [20]:
# Check if passages are actually relevant
for i in range(10):
    question = rag_df.iloc[i]['question']
    retrieved = retriever.get_relevant_documents(question, k=5)
    gold_answer = rag_df.iloc[i]['reference']
    
    print(f"Q: {question}")
    print(f"Gold: {gold_answer}")
    for j, doc in enumerate(retrieved):
        print(f"  [{j}] {doc.page_content[:150]}...")
    print()

C:\Users\akhil\AppData\Local\Temp\ipykernel_19568\4117566048.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved = retriever.get_relevant_documents(question, k=5)


Q: In what country is Normandy located?
Gold: France
  [0] Normandy...

Q: When were the Normans in Normandy?
Gold: 10th and 11th centuries
  [0] Normandy...
  [1] When Harry Met Sally......

Q: From which countries did the Norse originate?
Gold: Denmark, Iceland and Norway
  [0] Vikings...
  [1] Old Norse...

Q: Who was the Norse leader?
Gold: Rollo
  [0] Vikings...
  [1] Old Norse...

Q: What century did the Normans first gain their separate identity?
Gold: 10th century
  [0] Anglo-Saxons...
  [1] 11th century...
  [2] 4th century...
  [3] 20th century...
  [4] 14th century...

Q: Who gave their name to Normandy in the 1000's and 1100's
Gold: unanswerable
  [0] Normandy...

Q: What is France a region of?
Gold: unanswerable
  [0] History of France...
  [1] Yes, Virginia, there is a Santa Claus...

Q: Who did King Charles III swear fealty to?
Gold: unanswerable
  [0] Charles I of England...
  [1] Doctor Who...

Q: When did the Frankish identity emerge?
Gold: unanswerable
  [0] Charlema